In [ ]:
import fastf1
import pandas as pd
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

fastf1.Cache.enable_cache("../data/cache")

In [ ]:
schedule_2023 = fastf1.get_event_schedule(2023)

all_races_2023 = []

for _, event in schedule_2023.iterrows():
    try:
        race_name = event["EventName"]

        session = fastf1.get_session(2023, race_name, "R")
        session.load()

        race_results = session.results[[
            "Abbreviation",
            "TeamName",
            "GridPosition",
            "Position",
            "Status"
        ]].copy()

        race_results["RaceName"] = race_name
        race_results["RoundNumber"] = event["RoundNumber"]
        race_results["Year"] = 2023

        all_races_2023.append(race_results)

        print(f"Successfully loaded {race_name}")
    except Exception as e:

        print(f"Error loading {race_name}: {e}")

df_2023 = pd.concat(all_races_2023, ignore_index=True)

# df_2023.to_csv("../data/f1_2023_clean.csv", index=False)
# print("Data saved into 2023 csv")


In [ ]:
schedule_2024 = fastf1.get_event_schedule(2024)

all_races_2024 = []

for _, event in schedule_2024.iterrows():
    try:
        race_name = event["EventName"]

        session = fastf1.get_session(2024, race_name, "R")
        session.load()

        race_results = session.results[[
            "Abbreviation",
            "TeamName",
            "GridPosition",
            "Position",
            "Status"
        ]].copy()

        race_results["RaceName"] = race_name
        race_results["RoundNumber"] = event["RoundNumber"]
        race_results["Year"] = 2024

        all_races_2024.append(race_results)

        print(f"Successfully loaded {race_name}")
    except Exception as e:

        print(f"Error loading {race_name}: {e}")

df_2024 = pd.concat(all_races_2024, ignore_index=True)

# df_2024.to_csv("../data/f1_2024_clean.csv", index=False)
# print("Data saved into 2024 csv")

In [ ]:
# to avoid making features manually we can make a function for it

def add_features(df):
    df = df[df["Status"] == "Finished"].copy()

    df = df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

    df["PreviousPosition"] = (
        df.groupby("Abbreviation")["Position"]
        .shift(1)
    )

    df["Rolling3Average"] = (
        df.groupby("Abbreviation")["Position"]
        .rolling(window=3)
        .mean()
        .reset_index(0, drop=True)
    )

    df = df.dropna(subset=["PreviousPosition", "Rolling3Average"])

    return df

In [ ]:
df_2023 = add_features(df_2023)
df_2024 = add_features(df_2024)

In [ ]:
df_2023["PositionChange"] = df_2023["GridPosition"] - df_2023["Position"]
df_2024["PositionChange"] = df_2024["GridPosition"] - df_2024["Position"]

df_2023 = df_2023.sort_values(by=["Abbreviation", "RoundNumber"])
df_2024 = df_2024.sort_values(by=["Abbreviation", "RoundNumber"])

In [ ]:
df_2023["AveragePositionChange"] = (
    df_2023.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).mean())
)

df_2024["AveragePositionChange"] = (
    df_2024.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).mean())
)

df_2023["AveragePositionChange"] = df_2023["AveragePositionChange"].fillna(0)
df_2024["AveragePositionChange"] = df_2024["AveragePositionChange"].fillna(0)

In [ ]:
df_2023["Consistency"] = (
    df_2023.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).std())
)

df_2024["Consistency"] = (
    df_2024.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).std())
)

df_2023["Consistency"] = df_2023["Consistency"].fillna(0)
df_2024["Consistency"] = df_2024["Consistency"].fillna(0)


In [ ]:
df_2023["GridvsForm"] = df_2023["GridPosition"] - df_2023["Rolling3Average"]
df_2024["GridvsForm"] = df_2024["GridPosition"] - df_2024["Rolling3Average"]
# since rolling 3 average is very dominating
# we can use that feature to make new features
# we can check the grid position against the form of the driver 
# this will let us see if the driver is overperforming or underperforming compared to their recent form

df_2023["FormTrend"] = df_2023["Rolling3Average"] - df_2023["PreviousPosition"]
df_2024["FormTrend"] = df_2024["Rolling3Average"] - df_2024["PreviousPosition"]

features = [
    "TeamName",
    "GridPosition",
    "PreviousPosition",
    "AveragePositionChange",
    "Consistency",
    "GridvsForm",
    "FormTrend",
    "Position"
]

train_2023 = df_2023[features].copy()
test_2024 = df_2024[features].copy()

combined_df = pd.concat([train_2023, test_2024], ignore_index=True)
combined_df = pd.get_dummies(combined_df, columns=["TeamName"])

train_encoded = combined_df.iloc[:len(train_2023)]
test_encoded = combined_df.iloc[len(train_2023):]

X_train = train_encoded.drop("Position", axis=1)
y_train = train_encoded["Position"]

X_test = test_encoded.drop("Position", axis=1)
y_test = test_encoded["Position"]

print("Rolling3Average" in X_train.columns)
print(X_train.columns)



In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

mean_error = mean_absolute_error(y_test, prediction)
print(f"Mean Absolute Error: {mean_error:.2f}")

# HERE WE REMOVED THE ROLLING 3 AVERAGE FEATURE
# BECAUSE THAT FEATURE BASICALLY DOMINATED THE ENTIRE MODEL
# NOW THE ERROR HAS INCREASED BUT WE CAN SEE OTHER FEATURES HAVE MORE IMPORTANCE NOW

In [ ]:
test_info = df_2024[["Abbreviation", "TeamName", "RaceName", "RoundNumber", "Position", "GridPosition", "AveragePositionChange"]].copy()

In [ ]:
fantasy_table = test_info.copy()

fantasy_table["Predicted"] = prediction
fantasy_table["PredictedRounded"] = fantasy_table["Predicted"].round().astype(int)
# forgot to explain earlier, here we round up the predicted position
# because value we get is a continuous value and we want to convert it to a discrete position

fantasy_table["Confidence"] = abs(fantasy_table["Predicted"] - fantasy_table["PredictedRounded"])
# THIS SHOWS US HOW CERTAIN OUR PREDICTION IS
# CERTAINTY ABOUT THE DRIVER'S FINISHING POSITION
# WE NEED A VERY LOW SCORE FOR IT TO HAVE VERY HIGH CONFIDENCE
# Something that is as close to 0 as possible means its good, this shows that the model is CONFIDENT that the driver is around this position 

fantasy_table["Error"] = (fantasy_table["Position"] - fantasy_table["Predicted"]).abs()

fantasy_table = fantasy_table.sort_values(by="Predicted")

fantasy_table.head(20)

# this table is what we will use for the fantasy predictor

In [ ]:
# Now we try something different
# We try to pick a race and then see what the predictions are

race_name = "Dutch Grand Prix"

race_predictions = fantasy_table[
    fantasy_table["RaceName"] == race_name
].copy()

race_predictions.sort_values(by="Position").head(20)

# Here we provide the race name and use the fantasy table to see the predictions
# And then we sort by the position cuz i believe its easier
# And there is pretty good correlation between the predicted and actual positions

In [ ]:
safe_picks = race_predictions.sort_values(by="Predicted").head(3)

print(safe_picks[["Abbreviation", "TeamName", "Position", "Predicted", "PredictedRounded"]])

In [ ]:
midfield = race_predictions[
    (race_predictions["Predicted"] > 5) &
    (race_predictions["Predicted"] <= 12)
].copy()


midfield["GridGap"] = midfield["GridPosition"] - midfield["Predicted"]
# New feature that we are adding into v3 and we check if the driver finished better or worse than they started

midfield["GainerScore"] = (
   0.6 *  midfield["AveragePositionChange"]
    - 0.2 * midfield["Predicted"]
    - 0.1 * midfield["Confidence"]
    + 0.4 * midfield["GridGap"]
)
# This gainerscore is vital as it is the decision brain
# AveragePositionChange -> we check the amount of the position gained/lost
# Predicted is the expected finishing position, so lower = better, 
# Confidence controls the risk, high conf = high uncertainty, therefore we penalize the unstable predictions
# THIS FORMULA IS HEURISTIC
# The weights will determine the behavior,
# If predicted penalty increases it picks safer drivers
# If confidence penalty increases it avoids uncertain picks
# PositionChange increase, then picks high upside drivers

midfield = midfield[
    midfield["Confidence"] < 0.5
]

# midfield_positive = midfield[midfield["GainerScore"] > 0]

# if len(midfield_positive) >= 3:
#     position_gains_v2 = midfield_positive.sort_values(
#     by="GainerScore",
#     ascending=False
#     ).head(3)
# else:
#     position_gains_v2 = midfield.sort_values(
#         by="GainerScore",
#         ascending=False
#     ).head(3)

stable_midfield = midfield.sort_values(
    by="GainerScore",
    ascending=False
).head(2)

risky_pool = midfield[
    ~midfield["Abbreviation"].isin(stable_midfield["Abbreviation"])
]
# This takes all the midfield drivers except the ones in stable midfield, isin checks it
# tilde is meant as NOT

risky_pick = risky_pool.sort_values(
    by="AveragePositionChange",
    ascending=False
).head(1)


position_gains_v3 = pd.concat([
    stable_midfield,
    risky_pick
])

position_gains_v3[[
    "Abbreviation",
    "TeamName",
    "GridPosition",
    "GridGap",
    "Position",
    "Predicted",
    "PredictedRounded",
    "AveragePositionChange",
    "Confidence",
    "GainerScore"
]]

In [ ]:
fantasy_team_v3 = pd.concat([
    safe_picks,
    position_gains_v3
]).drop_duplicates()

fantasy_team_v3[[
    "Abbreviation",
    "TeamName",
    "Position",
    "Predicted",
    "PredictedRounded",
    "AveragePositionChange",
    "Confidence",
    "GainerScore"
]]

In [ ]:
def evaluate_fantasy_picks(fantasy_team):
    total_picks = len(fantasy_team)

    top5_hits = (fantasy_team["Position"] <= 5).sum()
    top10_hits = (fantasy_team["Position"] <= 10).sum()
    average_actual_finish = fantasy_team["Position"].mean()
    average_error = (abs(fantasy_team["Position"] - fantasy_team["Predicted"])).mean()

    return {
        "Total picks": int(total_picks),
        "Top 5 picks": int(top5_hits),
        "Top 5 hit rate": round(top5_hits / total_picks, 2), 
        "Top 10 picks": int(top10_hits),
        "Top 10 hit rate": round(top10_hits / total_picks, 2),
        "Average Actual Finish": round(average_actual_finish, 2),
        "Average Prediction Error": round(average_error, 2)
    }

In [ ]:
race_results_v3 = evaluate_fantasy_picks(fantasy_team_v3)

race_results_v3["RaceName"] = race_name
race_results_v3["Picks"] = ", ".join(fantasy_team_v3["Abbreviation"].tolist())

race_results_v3

In [ ]:
file_path = "../results/fantasy_v3_results.csv"

race_results_v2_df = pd.DataFrame([race_results_v3])

if os.path.exists(file_path):
    existing_df = pd.read_csv(file_path)
    existing_df = existing_df[existing_df["RaceName"] != race_name]

    updated_df = pd.concat([existing_df, race_results_v2_df], ignore_index=False)
    updated_df.to_csv(file_path, index=False)
else:
    race_results_v2_df.to_csv(file_path, index=False)